# EverFlavor AI - Food Taxonomy and Map Coverage

**Team:** EverGlow | **Course:** ITAI 2277 | **Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: new (2026-11-17).** Runs in under a minute, no downloads. Needs notebook 01 (recipes) and notebook 03
> (places).

---

## Purpose

Explore by place walks the Food Taxonomy (`data/reference/food_taxonomy.csv`: world, zone, food region, country,
state, culinary region), and the map shows the stores and restaurants of the pilot area. This notebook shows where
the app is thin, so the team knows what to add next.

| Section | What it does | Output |
|---|---|---|
| 2 | Recipes per branch of the tree; recipe countries and regions the tree does not have; branches with no recipes | printed |
| 3 | For each country: recipes, and places on the map whose cuisine or name says that country | `data/processed/taxonomy_map_coverage.csv` |
| 4 | The thin spots: many recipes and no places, places and few recipes | printed |

**Rules:** only counts; nothing is guessed. A place counts for a country only by its OpenStreetMap cuisine tag or the
cuisine word in its name, the same evidence the map shows.

---

## 1. Setup

The same setup as the other notebooks: it works unchanged in Google Colab, VS Code and Antigravity.

In [1]:
# %pip (not !pip) installs into the Python that runs this notebook
%pip install -q pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.util
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
IN_COLAB = importlib.util.find_spec("google.colab") is not None   # only exists in Colab


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "agent_tools", "calories", "charts", "checks", "confidence", "cooking", "cooking_time", "dish_steps", "crew",
                      "cuisine", "diets", "dish_knowledge", "dish_names", "environment", "evaluation", "features", "flags", "freshness", "ingredients",
                      "knowledge", "llm", "nutrition", "nutrition_quality", "parsing", "pipeline", "places", "preferences", "preparation", "recipe_details",
                      "progress", "recommend", "refresh", "reporting", "review", "safety", "seasons", "sources",
                      "saved_recipes", "stores", "variants", "waiter", "recipe_check", "restrictions", "taxonomy"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

OUT = Path("data/processed")
print(f"Running from {short_path(PROJECT_ROOT)} ({'Colab' if IN_COLAB else 'local'})")

Running from ~\Downloads\jupiter-exploration\The-EverFlavor-AI (local)


---

## 2. Recipes in the tree

In [3]:
from everflavor.taxonomy import load_taxonomy

tax = load_taxonomy()
recipes = pd.concat([pd.read_parquet(f"data/processed/recipes_{split}.parquet",
                                     columns=["recipe_id", "origin_country", "origin_region"])
                     for split in ("train", "val", "test")], ignore_index=True)
counts = tax.counts(recipes)
print(f"{len(recipes):,} recipes; {len(tax.nodes)} places in the tree")
for zone in tax.children("world"):
    print(f"  {zone.name}: {counts[zone.slug]:,}")

291,755 recipes; 150 places in the tree
  Asia: 19,720
  Europe: 50,478
  Middle East & North Africa: 3,531
  Africa: 2,597
  North America: 147,233
  Latin America: 12,327
  Caribbean: 909
  Oceania: 3,187


In [4]:
known_countries = {n.name for n in tax.nodes.values() if n.level == "country"}
known_regions = {n.name for n in tax.nodes.values() if n.level in ("division", "state", "culinary_region")}
by_country = recipes["origin_country"].value_counts()
missing_countries = by_country[[c not in known_countries and c != "Unknown" for c in by_country.index]]
by_region = recipes.dropna(subset=["origin_region"]).groupby(["origin_country", "origin_region"]).size()
missing_regions = by_region[[r not in known_regions for _, r in by_region.index]].sort_values(ascending=False)
empty = sorted(n.name for n in tax.nodes.values() if counts[n.slug] == 0 and n.level != "world")
print(f"Recipe countries the tree does not have ({len(missing_countries)}):", missing_countries.head(15).to_dict())
print(f"Recipe regions the tree does not have ({len(missing_regions)}):", missing_regions.head(15).to_dict())
print(f"Branches with no recipes ({len(empty)}):", empty[:30])

Recipe countries the tree does not have (0): {}
Recipe regions the tree does not have (0): {}
Branches with no recipes (0): []


---

## 3. Recipes and places, country by country

A place counts for a country when its OpenStreetMap cuisine tag is that country's cuisine word (`chinese`,
`mexican`...) or its name carries the word (`name_hint`). Shops without a cuisine count for none.

In [5]:
from everflavor.stores import COUNTRY_CUISINES

places = pd.read_parquet("data/processed/places.parquet", columns=["osm_id", "category", "cuisine", "name_hint"])


def words(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return set()
    items = value if not isinstance(value, str) else [value]
    return {str(w).strip().lower() for w in items if str(w).strip()}


place_words = [words(c) | words(h) for c, h in zip(places["cuisine"], places["name_hint"])]
rows = []
for country, word in COUNTRY_CUISINES.items():
    node = tax.node_for(country)
    zone = tax.ancestor(node, "zone").name if node and tax.ancestor(node, "zone") else ""
    region = tax.ancestor(node, "region").name if node and tax.ancestor(node, "region") else ""
    rows.append({"zone": zone, "food_region": region, "country": country, "cuisine_word": word,
                 "recipes": int(by_country.get(country, 0)),
                 "places": sum(word in w for w in place_words)})
coverage = pd.DataFrame(rows).sort_values(["zone", "food_region", "country"])
coverage.to_csv(OUT / "taxonomy_map_coverage.csv", index=False)
print(f"Saved {short_path(OUT / 'taxonomy_map_coverage.csv')}: {len(coverage)} countries")
coverage.groupby("zone")[["recipes", "places"]].sum()

Saved data\processed\taxonomy_map_coverage.csv: 29 countries


,recipes,places
zone,,
Africa,110,3
Asia,19517,568
Caribbean,801,6
Europe,45639,171
Latin America,11987,404
Middle East & North Africa,3284,24
North America,131162,316


---

## 4. The thin spots

**Recipes but no places:** people can cook it, but the map has nowhere to show for it. **Places but few
recipes:** the map shows it, but the recipe side has little.

In [6]:
no_places = coverage[(coverage["recipes"] >= 50) & (coverage["places"] == 0)].sort_values("recipes", ascending=False)
few_recipes = coverage[(coverage["places"] >= 5) & (coverage["recipes"] < 50)].sort_values("places", ascending=False)
print("Recipes (50+) but no places:")
print(no_places[["country", "recipes"]].to_string(index=False) if len(no_places) else "  none")
print()
print("Places (5+) but fewer than 50 recipes:")
print(few_recipes[["country", "places", "recipes"]].to_string(index=False) if len(few_recipes) else "  none")

Recipes (50+) but no places:
  country  recipes
Indonesia      218
    Egypt      206

Places (5+) but fewer than 50 recipes:
  none


---

## Summary

- Section 2 lists the recipe countries and regions to add to the tree (a row each in `food_taxonomy.csv`, with its
  parent), and the branches still waiting for recipes.
- Sections 3 and 4 show where the map and the recipes disagree; the pilot area is Greater Houston, so a country with
  no places there may simply have none nearby.